## Setup

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv
import xarray as xr

In [ ]:
load_dotenv()

RAW_PATH = os.getenv('XRAY_RAW_PATH')
PROCESSED_PATH = os.getenv('XRAY_PROCESSED_PATH')
TREATED_PATH = os.getenv('XRAY_TREATED_PATH')
DATA_YEARS = range(2010, 2025)

## Read Files and create Intermediate CSVs

In [ ]:
for y_ in tqdm(DATA_YEARS, desc="Processando Anos (Raios-X)"):
    y_str = str(y_)
    output_dir = os.path.join(PROCESSED_PATH, y_str)
    os.makedirs(output_dir, exist_ok=True)

    output_csv = os.path.join(output_dir, f'xrays_1m_{y_str}.csv')

    if os.path.exists(output_csv):
        continue

    xray_files = glob.glob(os.path.join(RAW_PATH, y_str, 'sci_xrsf-l2-avg1m_*.nc'))

    df_xray_list = []
    for f in xray_files:
        try:
            ds = xr.open_dataset(f)
            df = ds.to_dataframe().reset_index()
            df_xray_list.append(df)
            ds.close()
        except Exception as e:
            print(f"Erro ao ler NC (X-Rays): {f} - {e}")

    if df_xray_list:
        df_year = pd.concat(df_xray_list, ignore_index=True)
        df_year.to_csv(output_csv, index=False)

## Reading Intermediate CSVs
Harmonização de Schemas L1b/L2 e Desduplicação Temporal

> 🛡️ **Nota de Arquitetura: Seleção de Satélites Primários**
>
> A desduplicação (`drop_duplicates(subset=['time'])`) aplicada neste passo atua apenas sobre possíveis sobreposições marginais de leitura. **Não há risco de mesclagem cega entre satélites primários e secundários.**
>
> Conforme as diretrizes de qualidade do artigo *Defects and Inconsistencies in Solar Flare Data Sources*, a restrição estrita aos dados do satélite principal (GOES-13, 14, 15 ou 16) já foi garantida *upstream*, durante a etapa de extração (`xrays_nc_scrape.ipynb`). Lá, o mapeamento de datas seguiu rigorosamente a cronologia oficial de operação primária da NCEI, rejeitando o download de qualquer telemetria secundária para um dado dia.

In [ ]:
PROCESSED_XRAY_DATA = {}
core_columns = ['time', 'xrsb_flux', 'xrsb_flag']

df_harmonized_list = []

for year in tqdm(DATA_YEARS, desc="Carregando e Harmonizando CSVs"):
    y_str = str(year)
    file_path = os.path.join(PROCESSED_PATH, y_str, f'xrays_1m_{y_str}.csv')

    if os.path.exists(file_path):
        try:
            df_raw = pd.read_csv(file_path)

            # 1. Harmonização e Isolamento do Fluxo Primário
            # Se o satélite for GOES-R (possui 'quad_diode'), isolamos a medição primária consolidada
            if 'quad_diode' in df_raw.columns:
                df_filtered = df_raw[df_raw['quad_diode'] == 0].copy()
            else:
                df_filtered = df_raw.copy()

            cols_present = [c for c in core_columns if c in df_filtered.columns]
            df_harmonized = df_filtered[cols_present].copy()

            # --- INÍCIO DA ADEQUAÇÃO PARA PANDAS 3.0 ---
            # Isola a conversão de datetime em uma Series temporária
            time_series = pd.to_datetime(df_harmonized['time'])

            # Aplica o fuso horário (UTC) na Series temporária
            if time_series.dt.tz is None:
                time_series = time_series.dt.tz_localize('UTC')
            else:
                time_series = time_series.dt.tz_convert('UTC')

            # Usa o .assign() para recriar o DataFrame com a coluna 'time' atualizada
            df_harmonized = df_harmonized.assign(time=time_series)
            # --- FIM DA ADEQUAÇÃO ---

            df_harmonized = df_harmonized.drop_duplicates(subset=['time']).sort_values('time')

            df_harmonized_list.append(df_harmonized)

        except pd.errors.EmptyDataError:
            print(f"Aviso: Arquivo CSV vazio para o ano {y_str}.")
    else:
        print(f"Aviso: Arquivo CSV não encontrado para o ano {y_str}.")

# Concatenação final de toda a série temporal
if df_harmonized_list:
    df_xrays_full = pd.concat(df_harmonized_list, ignore_index=True)
    # Desduplicação global para tratar eventuais sobreposições nas viradas de ano
    df_xrays_full = df_xrays_full.drop_duplicates(subset=['time']).sort_values('time').reset_index(drop=True)
    print(f"Total de medições (1 min): {len(df_xrays_full)}")
else:
    df_xrays_full = pd.DataFrame()
    print("\nFalha: Nenhum dado foi processado.")

In [ ]:
display(df_xrays_full.head())

## Treating Data

### 1. Filtro de Qualidade e Limpeza Instrumental

Limpeza inicial baseada nos defeitos relatados na literatura para os dados espaciais de fluxo de Raios-X (*Science-Quality L2*). A base é filtrada para descartar medições comprometidas por anomalias de telemetria e ruído instrumental extremo (valores fisicamente impossíveis).

**Atenção Metodológica:** Diferente dos magnetogramas espaciais (onde linhas defeituosas são excluídas com `.dropna()`), na série temporal contínua de raios-X os valores corrompidos são **substituídos por `NaN`**. Isso é imperativo para preservar a integridade geométrica do *grid* temporal de 1 minuto, garantindo que a reamostragem (12 minutos) do próximo passo ocorra sem distorções de janela.

In [ ]:
df_xrays_clean = df_xrays_full.copy()
initial_len = len(df_xrays_clean)

In [ ]:
# 1.1 FILTRO DE QUALIDADE (QUALITY FLAG NCEI)
# Na padronização Science-Quality, a flag 0 indica dado nominal (sem anomalias). Qualquer valor diferente de 0 sinaliza calibrações, eclipses ou corrupção.
mask_invalid_flag = df_xrays_clean['xrsb_flag'] != 0
len_bad_quality = mask_invalid_flag.sum()

In [ ]:
# 1.2 FILTRO DE INTEGRIDADE FÍSICA
# O ruído instrumental no vácuo espacial pode gerar leituras de fluxo <= 0. Um fluxo negativo/zerado é fisicamente impossível e corrompe escalas logarítmicas.
# Nota: NaNs pré-existentes na base são ignorados por essa máscara
mask_unphysical_flux = df_xrays_clean['xrsb_flux'] <= 0
len_unphysical = mask_unphysical_flux.sum()

In [ ]:
# 1.3 APLICAÇÃO DA MÁSCARA E PRESERVAÇÃO DA GRADE TEMPORAL
# Substituicao por NaN em vez de dropar a linha, mantendo a coluna 'time' intacta.
# Mantemos apenas as colunas vitais para otimizar a memória RAM a partir daqui.
mask_corrupted = mask_invalid_flag | mask_unphysical_flux
df_xrays_clean.loc[mask_corrupted, 'xrsb_flux'] = np.nan

df_xrays_clean = df_xrays_clean[['time', 'xrsb_flux']]
valid_count = initial_len - mask_corrupted.sum()

In [ ]:
print("--- Resumo do Passo 1: Filtro de Qualidade (Raios-X) ---")
print(f"Total de medições contínuas analisadas: {initial_len}")
print(f"Anomalias de Flag Operacional (xrsb_flag != 0): {len_bad_quality}")
print(f"Anomalias Físicas Instrumentais (fluxo <= 0): {len_unphysical}")
print(f"Total de ruído mascarado com NaN: {mask_corrupted.sum()}")
print(f"Taxa de integridade dos dados: {(valid_count / initial_len) * 100:.2f}%")
print(f"Dimensão da base limpa: {df_xrays_clean.shape}")

### 2. Sincronização Magnética (Early Resampling - 12 min)

O alinhamento temporal entre a série global de raios-X e as medições regionais de magnetogramas (SHARPs) é essencial para garantir a sanidade da matriz de *features* na arquitetura Solarfall. O dataset original de 1 minuto é condensado para janelas estritas de 12 minutos (ancoradas em `00:00`, `00:12`, etc.).

**Extração Bidimensional:**
* `xrsb_flux_max`: Preserva o pico impulsivo máximo dentro da janela. Fundamental para determinar a severidade do evento (flare) e classificar os rótulos de alvo.
* `xrsb_flux_mean`: Captura o estado inercial e o ruído de fundo da coroa solar. Permite aos modelos baseados em árvores diferenciarem elevações graduais de temperatura de explosões genuínas.


> 🛡️ **Nota de Arquitetura: Fluxo Global vs. Atribuição Regional (ARs)**
>
> As métricas extraídas aqui (`xrsb_flux_max` e `xrsb_flux_mean`) representam a irradiância **global** do disco solar, sem distinção espacial.
>
> A literatura adverte que a base *Science-Quality* carece da identificação das Regiões Ativas (ARs) causadoras das explosões, o que poderia degradar o desempenho dos modelos pela perda de amostras. Na arquitetura *Solarfall*, este problema está mitigado:
> * O cruzamento do fluxo global com as ARs correspondentes (SHARPs) **não ocorre neste script**.
> * A recuperação e atribuição das Regiões Ativas às explosões globais é resolvida em um pipeline dedicado (`events_treatment.ipynb`), que utiliza um algoritmo em cascata (catálogos SWPC/SSW) e inferência geométrica (Stonyhurst para HPC via distâncias euclidianas) para garantir que o pico de fluxo seja associado à assinatura magnética correta.

In [ ]:
# Ao calcular o 'max' e 'mean', o pandas automaticamente ignora os NaNs inseridos no Passo 1. Se uma janela de 12 minutos for composta inteiramente por NaNs, o resultado será NaN.

# Define 'time' como índice para otimizar o resample
df_xrays_indexed = df_xrays_clean.set_index('time')

# Executa o resample estrito para 12 minutos
df_xrays_12m = df_xrays_indexed.resample('12min').agg(
    xrsb_flux_max=('xrsb_flux', 'max'),
    xrsb_flux_mean=('xrsb_flux', 'mean')
).reset_index()

In [ ]:
print("--- Resumo do Passo 2: Reamostragem ---")
print(f"Dimensão da base original (1 minuto): {len(df_xrays_clean)}")
print(f"Dimensão da base sincronizada (12 minutos): {len(df_xrays_12m)}")
print(f"Taxa de compressão temporal: {len(df_xrays_clean) / len(df_xrays_12m):.2f}x")

display(df_xrays_12m.head())

### 3. Tratamento de Gaps e Criação de Runs Temporais

A literatura enfatiza a necessidade de avaliar a continuidade temporal das observações, particionando os dados em *runs* contínuas. Como explosões em raios-X possuem dinâmicas muito rápidas, preencher longos buracos sem dados criaria rampas artificiais de energia.

**Regras aplicadas:**
1. **Fluxo Basal (`xrsb_flux_mean`):** Permitida a interpolação linear para *gaps* pequenos e isolados (máximo de 1 janela de 12 min).
2. **Pico Impulsivo (`xrsb_flux_max`):** Nenhuma interpolação é aplicada. Fica estritamente como `NaN` para atuar como *firewall* contra falsos decaimentos.
3. **Quebra de Run (`run_id`):** Gaps superiores a 1 janela rompem a continuidade temporal. Atribui-se um novo identificador (`run_id`) para o bloco temporal subsequente.

> 🛡️ **Nota de Arquitetura: Omissão de Filtros de Borda Solar (Limb Filtering) no Fluxo XRS**
>
> O artigo *Defects and Inconsistencies* alerta sobre a corrupção severa de dados magnéticos localizados a mais de $\pm70^\circ$ do meridiano central (efeito de projeção).
>
> **Nenhum filtro espacial é aplicado a esta série temporal de raios-X.** Como o sensor XRS captura a energia de forma integrada (todo o disco solar), remover dados globais com base na localização da explosão criaria falhas e *gaps* artificiais indesejados no *background flux*. A filtragem de borda ($\pm70^\circ$) é aplicada **estritamente sobre a matriz de preditores regionais (SHARPs)** durante a engenharia de features, garantindo que o modelo apenas aprenda com campos magnéticos observados frontalmente.

In [ ]:
df_xrays_runs = df_xrays_12m.copy()

# Conta NaNs antes da interpolação para auditoria
nans_mean_before = df_xrays_runs['xrsb_flux_mean'].isna().sum()
nans_max_before = df_xrays_runs['xrsb_flux_max'].isna().sum()

In [ ]:
# 3.1 INTERPOLAÇÃO RESTRITA DA MÉDIA (BACKGROUND FLUX)
# Preenche linearmente buracos isolados (limit=1 significa no máximo 12 min ausentes)
df_xrays_runs.loc[:, 'xrsb_flux_mean'] = df_xrays_runs['xrsb_flux_mean'].interpolate(method='linear', limit=1)

# Conta os NaNs recuperados
nans_mean_after = df_xrays_runs['xrsb_flux_mean'].isna().sum()
gaps_interpolated = nans_mean_before - nans_mean_after

In [ ]:
# 3.2 IDENTIFICAÇÃO DOS GAPS INSUPRÍVEIS E QUEBRA DE RUNS
# Qualquer NaN sobrevivente na média indica um buraco maior que o tolerado (>= 24 min).
insurmountable_gap = df_xrays_runs['xrsb_flux_mean'].isna()
is_valid = ~insurmountable_gap

# Uma nova run começa se a linha atual for válida E a linha anterior for um gap. (fill_value=True garante que o primeiro registro válido do dataset inicie a Run 1)
new_run_starts = is_valid & insurmountable_gap.shift(1, fill_value=True)

# O cumsum() cria um ID numérico que só incrementa quando uma nova run começa
df_xrays_runs.loc[:, 'run_id'] = new_run_starts.cumsum()

# Nas linhas que compõem o próprio gap, run_id é vazio (NaN/pd.NA)
df_xrays_runs.loc[insurmountable_gap, 'run_id'] = pd.NA

In [ ]:
# 3.3 CONSOLIDAÇÃO DO DATAFRAME
# Convertendo run_id para o tipo Integer que suporta NaNs no Pandas ('Int64')
df_xrays_runs = df_xrays_runs.astype({'run_id': 'Int64'})

# Reordena colunas para facilitar leitura
df_xrays_runs = df_xrays_runs[['run_id', 'time', 'xrsb_flux_max', 'xrsb_flux_mean']]

total_runs = df_xrays_runs['run_id'].max()
valid_windows = is_valid.sum()
gap_windows = insurmountable_gap.sum()

In [ ]:
print("--- Resumo do Passo 3: Gaps e Runs ---")
print(f"Total de janelas de 12 min na matriz: {len(df_xrays_runs)}")
print(f"Pequenos gaps interpolados na média (1 janela): {gaps_interpolated}")
print(f"Janelas rotuladas como buracos intransponíveis (>= 2 janelas): {gap_windows}")
print(f"Total de Runs contínuas válidas criadas: {total_runs}")
print(f"Os picos máximos (xrsb_flux_max) foram mantidos intocados com {nans_max_before} NaNs.")

In [ ]:
display(df_xrays_runs[df_xrays_runs['run_id'].isna()].head()) # Exemplo de buracos
display(df_xrays_runs.head())

### 4. Preservação da Escala Física Nativa e Consolidação (Trusted Layer)

Os dados operacionais anteriores ao GOES-16 possuíam um fator de escala artificial de 0.7 aplicado pela SWPC. Como nossa fonte é o produto *Science-Quality L2* da NCEI, as irradiâncias XRS já estão corrigidas e fornecidas na unidade física real de $W/m^2$, sem os offsets da SWPC.

**Ação Final:** Nenhuma retrocorreção matemática é necessária. O dataset consolidado, com as *runs* temporais definidas e na cadência de 12 minutos, é exportado para a camada *Trusted*, pronto para o *Feature Engineering* e integração com os magnetogramas.

In [ ]:
df_final = df_xrays_runs.copy()

In [ ]:
# 4.1 ORDENAÇÃO E FORMATAÇÃO FINAL
# Garantir que a base esteja perfeitamente cronológica e com tipos corretos
df_final = df_final.sort_values(by='time').reset_index(drop=True)

# Arredondar os fluxos para evitar imprecisões de ponto flutuante excessivas
df_final.loc[:, 'xrsb_flux_max'] = df_final['xrsb_flux_max'].astype('float32')
df_final.loc[:, 'xrsb_flux_mean'] = df_final['xrsb_flux_mean'].astype('float32')

In [ ]:
df_final

## Exporting Data

**Exportação em Parquet:**
O dataset consolidado, com as *runs* temporais definidas e na cadência de 12 minutos, é exportado no formato `.parquet`. O uso do Parquet preserva rigorosamente a tipagem dos dados (especialmente o tipo `Int64` da coluna `run_id`, garantindo que os valores nulos `pd.NA` não sejam convertidos erroneamente para *float* na leitura futura).

In [ ]:
os.makedirs(TREATED_PATH, exist_ok=True)
output_filepath = os.path.join(TREATED_PATH, 'treated_xray.parquet')

df_final.to_parquet(output_filepath, index=False)

In [ ]:
print("✅ Pipeline de Tratamento de Raios-X Finalizado com Sucesso!")
print(f"Dataset exportado para: {output_filepath}")
print(f"Dimensões finais: {df_final.shape}")
print(f"Total de Runs: {df_final['run_id'].max(skipna=True)}")
print("\nAmostra dos dados finais (Treated):")
display(df_final.head(10))